In [1]:
%pip install -q dotenv llama_stack_client==0.4.2

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

from llama_stack_client import LlamaStackClient

In [3]:
def stream_response(stream):
    """Stream responses API events with MCP tool call support."""
    for event in stream:
        event_type = getattr(event, "type", None)

        if event_type == "response.output_text.delta":
            print(event.delta, end="", flush=True)

        elif event_type == "response.refusal.delta":
            print(event.delta, end="", flush=True)

        # MCP tool call events
        elif event_type == "response.output_item.added":
            item = getattr(event, "item", None)
            if item and getattr(item, "type", None) == "mcp_call":
                print(f"\n🔧 MCP call: {getattr(item, 'name', '')}")
        elif event_type == "response.mcp_call.in_progress":
            print("  executing...")
        elif event_type == "response.mcp_call.completed":
            print("  completed")
        elif event_type == "response.mcp_call.failed":
            print("  failed")

    print()

In [4]:
load_dotenv()
base_url = os.getenv("REMOTE_BASE_URL", "http://localhost:8321")

client = LlamaStackClient(base_url=base_url)

# Get MCP server URLs from registered toolgroups
toolgroups = client.toolgroups.list()
mcp_tools = []
for tg in toolgroups:
    if tg.mcp_endpoint:
        label = tg.identifier.replace("mcp::", "")
        print(f"  Found MCP toolgroup: {tg.identifier} -> {tg.mcp_endpoint.uri}")
        mcp_tools.append({
            "type": "mcp",
            "server_label": label,
            "server_url": tg.mcp_endpoint.uri,
        })

/tmp/ipykernel_1587/238296959.py:7: DeprecationWarning: deprecated
  toolgroups = client.toolgroups.list()
INFO:httpx:HTTP Request: GET http://llamastack-distribution-service:8321/v1/toolgroups "HTTP/1.1 200 OK"


  Found MCP toolgroup: mcp::customer -> http://mcp-customer-service:9001/mcp
  Found MCP toolgroup: mcp::finance -> http://mcp-finance-service:9002/mcp


In [5]:
MODEL = "vllm/qwen3-8b"
INSTRUCTIONS = """You are a helpful customer service assistant. 
Use the available MCP tools to search customers, fetch orders and invoices.
Always return the response in a friendly and helpful tone."""

TOOLS = mcp_tools

In [6]:
question = "find more detail about franwilson@example.com"

stream = client.responses.create(
    model=MODEL,
    input=question,
    instructions=INSTRUCTIONS,
    tools=TOOLS,
    stream=True,
)

stream_response(stream)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/responses "HTTP/1.1 200 OK"


<think>
Okay, the user is asking for more details about the email address franwilson@example.com. Let me think about how to approach this.

First, I need to check which tools are available. The tools provided are search_customers, get_customer, fetch_order_history, and fetch_invoice_history. 

The user's query is about an email address, so the most relevant tool here is search_customers. That function allows filtering by contact_email. Since the email is provided, I should use that parameter. 

Wait, the search_customers function has parameters for company_name, contact_name, contact_email, and phone. The contact_email is optional and does support partial matching. So I can pass franwilson@example.com as the contact_email argument. 

By doing that, the function should return a list of customers who have that email. If there's a match, I can then get more details using get_customer with the customer_id. But first, I need to check if the email exists in the customers. 

Alternatively, ma

In [7]:
question = "find all order and invoice about franwilson@example.com"

stream = client.responses.create(
    model=MODEL,
    input=question,
    instructions=INSTRUCTIONS,
    tools=TOOLS,
    stream=True,
)

stream_response(stream)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-service:8321/v1/responses "HTTP/1.1 200 OK"


<think>
Okay, the user wants to find all orders and invoices related to the email address franwilson@example.com. Let me think about how to approach this.

First, I need to find the customer associated with that email. The available functions include search_customers, which can filter by contact_email. So I should call search_customers with contact_email set to franwilson@example.com. This will give me a list of customers matching that email.

Once I have the customer details, I can get their customer_id. Then, using that customer_id, I can fetch the order history and invoice history. The fetch_order_history and fetch_invoice_history functions both require a customer_id. 

Wait, but the user might have multiple customers with the same email? Although unlikely, maybe the search_customers will return multiple results. But since the email is unique, probably only one. Let me proceed step by step.

First, call search_customers with contact_email. If there's a match, extract the customer_id